In [1]:
# import dependencies
import pandas as pd
import numpy as np

## Load ACS

And extract ID and town name

In [2]:
# load ACS files
df_median_income = pd.read_csv(
    "../data/raw/acs/median_household_income/ACSDT5Y2024.B19013-Data.csv", skiprows=1
)
df_percent_elderly = pd.read_csv(
    "../data/raw/acs/sex_by_age/ACSDT5Y2024.B01001-Data.csv", skiprows=1
)
df_median_year_house_built = pd.read_csv(
    "../data/raw/acs/median_year_structure_built/ACSDT5Y2024.B25035-Data.csv",
    skiprows=1,
)
df_vehicle = pd.read_csv(
    "../data/raw/acs/household_vehicles/ACSDT5Y2024.B08201-Data.csv", skiprows=1
)
df_renter_occupied = pd.read_csv(
    "../data/raw/acs/renter_occupied/ACSDT5Y2024.B25003-Data.csv", skiprows=1
)
df_education_level = pd.read_csv(
    "../data/raw/acs/education_level/ACSDT5Y2024.B15003-Data.csv", skiprows=1
)
df_poverty_level = pd.read_csv(
    "../data/raw/acs/poverty_level/ACSDT5Y2024.B17001-Data.csv", skiprows=1
)
df_disability_status = pd.read_csv(
    "../data/raw/acs/disability_status/ACSDT5Y2024.B18101-Data.csv", skiprows=1
)
df_mobile_home = pd.read_csv(
    "../data/raw/acs/mobile_home/ACSDT5Y2024.B25024-Data.csv", skiprows=1
)

In [3]:
# list of all ACS dfs for easier processing
acs_dfs = [
    df_median_income,
    df_percent_elderly,
    df_median_year_house_built,
    df_vehicle,
    df_renter_occupied,
    df_education_level,
    df_poverty_level,
    df_disability_status,
    df_mobile_home,
]

In [4]:
# extract FIPS GEOID from last 10 digits of Geography column
# extract town name (everything before the first comma)
for df in acs_dfs:
    df["GEOID"] = df["Geography"].str[-10:]
    df["town_name"] = df["Geographic Area Name"].str.split(",", n=1).str[0]

## Process individual dataframes

Retain margin of error and some raw counts for future sensitivity analysis or eda. If calculated here, margin of error follows official ACS guidance from the Census Bureau.

In [5]:
# helper function to calculate margin of error (MOE)
def ratio_moe(numerator, numerator_moe, denominator, denominator_moe):
    """
    Calculate the margin of error (MOE) for a ratio or percent, using the ACS-recommended formula.

    MOE = 100 * sqrt((numerator_moe / denominator)^2 + ((numerator * denominator_moe) / denominator^2)^2)

    Based on U.S. Census Bureau ACS guidance. Assumes independence between numerator and denominator.
    Annoyingly complicated. Had to look it up. Statistics hurt my brain.
    """
    return 100 * np.sqrt(
        (numerator_moe / denominator) ** 2
        + ((numerator * denominator_moe) / (denominator**2)) ** 2
    )

In [6]:
# get median income df with only relevant columns
df_median_income = df_median_income[
    [
        "GEOID",
        "town_name",
        "Estimate!!Median household income in the past 12 months (in 2024 inflation-adjusted dollars)",
        "Margin of Error!!Median household income in the past 12 months (in 2024 inflation-adjusted dollars)",
    ]
].rename(
    columns={
        "Estimate!!Median household income in the past 12 months (in 2024 inflation-adjusted dollars)": "median_income",
        "Margin of Error!!Median household income in the past 12 months (in 2024 inflation-adjusted dollars)": "median_income_moe",
    }
)
df_median_income.head()

,GEOID,town_name,median_income,median_income_moe
0,5000100325,Addison town,106667,7181
1,5000108575,Bridport town,83214,17776
2,5000109025,Bristol town,74620,8110
3,5000116000,Cornwall town,114583,32017
4,5000126300,Ferrisburgh town,106989,21771


In [ ]:
# calculate percent elderly (65+) by summing relevant columns and dividing by total population


# helper to sum columns by prefix and age group
def sum_age_groups(df, prefix, age_groups):
    cols = [f"{prefix}{age}" for age in age_groups]
    return df[cols].sum(axis=1)


# define age groups for 65+
age_groups = [
    "65 and 66 years",
    "67 to 69 years",
    "70 to 74 years",
    "75 to 79 years",
    "80 to 84 years",
    "85 years and over",
]

# build full column names for estimates and MOEs
# intentionally hardcoding columns, to fail if ACS changes their column names in the future
male_est = [f"Estimate!!Total:!!Male:!!{age}" for age in age_groups]
female_est = [f"Estimate!!Total:!!Female:!!{age}" for age in age_groups]
male_moe = [f"Margin of Error!!Total:!!Male:!!{age}" for age in age_groups]
female_moe = [f"Margin of Error!!Total:!!Female:!!{age}" for age in age_groups]

elderly_cols = male_est + female_est
elderly_moe_cols = male_moe + female_moe

# rename total population columns for easier reference
df_percent_elderly = df_percent_elderly.rename(
    columns={
        "Estimate!!Total:": "total_population",
        "Margin of Error!!Total:": "total_population_moe",
    }
)

# calculate elderly raw counts and MOEs
df_percent_elderly["elderly_65plus"] = df_percent_elderly[elderly_cols].sum(axis=1)
# sum MOEs in quadrature (square root of sum of squares) for elderly population
df_percent_elderly["elderly_65plus_moe"] = np.sqrt(
    (df_percent_elderly[elderly_moe_cols] ** 2).sum(axis=1)
)

# calculate percent and MOE
df_percent_elderly["percent_elderly"] = (
    df_percent_elderly["elderly_65plus"] / df_percent_elderly["total_population"] * 100
)
# MOE_percent = 100 * sqrt( (elderly_moe/total)^2 + (elderly*total_population_moe/total^2)^2 )
df_percent_elderly["percent_elderly_moe"] = ratio_moe(
    df_percent_elderly["elderly_65plus"],
    df_percent_elderly["elderly_65plus_moe"],
    df_percent_elderly["total_population"],
    df_percent_elderly["total_population_moe"],
)

# keep only relevant columns, some renamed for clarity
df_percent_elderly = df_percent_elderly[
    [
        "GEOID",
        "town_name",
        "percent_elderly",
        "percent_elderly_moe",
        "elderly_65plus",
        "elderly_65plus_moe",
        "total_population",
        "total_population_moe",
    ]
]
df_percent_elderly.head()

,GEOID,town_name,percent_elderly,percent_elderly_moe,elderly_65plus,elderly_65plus_moe,total_population,total_population_moe
0,5000100325,Addison town,27.148936,6.322324,319,54.735729,1175,185
1,5000108575,Bridport town,24.291498,6.468954,300,62.665780,1235,204
2,5000109025,Bristol town,21.942827,4.833447,829,182.518492,3778,26
3,5000116000,Cornwall town,25.662252,6.107244,310,51.195703,1208,207
4,5000126300,Ferrisburgh town,23.299511,3.794391,620,100.891030,2661,17


In [8]:
# get median year house built, with relevant columns
df_median_year_house_built = df_median_year_house_built[
    [
        "GEOID",
        "town_name",
        "Estimate!!Median year structure built",
        "Margin of Error!!Median year structure built",
    ]
].rename(
    columns={
        "Estimate!!Median year structure built": "median_year_house_built",
        "Margin of Error!!Median year structure built": "median_year_house_built_moe",
    }
)
df_median_year_house_built.head()

,GEOID,town_name,median_year_house_built,median_year_house_built_moe
0,5000100325,Addison town,1980,6
1,5000108575,Bridport town,1978,6
2,5000109025,Bristol town,1976,5
3,5000116000,Cornwall town,1976,7
4,5000126300,Ferrisburgh town,1968,6


In [9]:
# extract vehicle availability (percent of households with no vehicle available)

# rename columns
df_vehicle = df_vehicle.rename(
    columns={
        "Estimate!!Total:!!No vehicle available": "no_vehicle_available",
        "Margin of Error!!Total:!!No vehicle available": "no_vehicle_available_moe",
        "Estimate!!Total:": "total_households",
        "Margin of Error!!Total:": "total_households_moe",
    }
)

# calculate percent of households with no vehicle available
df_vehicle["pct_no_vehicle"] = (
    df_vehicle["no_vehicle_available"] / df_vehicle["total_households"]
) * 100


# calculate MOE for percent no vehicle using ratio MOE formula
# MOE_percent = 100 * sqrt( (no_vehicle_moe/total)^2 + (no_vehicle*total_moe/total^2)^2 )
df_vehicle["pct_no_vehicle_moe"] = ratio_moe(
    df_vehicle["no_vehicle_available"],
    df_vehicle["no_vehicle_available_moe"],
    df_vehicle["total_households"],
    df_vehicle["total_households_moe"],
)

# keep only relevant columns
df_vehicle = df_vehicle[
    [
        "GEOID",
        "town_name",
        "pct_no_vehicle",
        "pct_no_vehicle_moe",
        "no_vehicle_available",
        "no_vehicle_available_moe",
        "total_households",
        "total_households_moe",
    ]
]
df_vehicle.head()

,GEOID,town_name,pct_no_vehicle,pct_no_vehicle_moe,no_vehicle_available,no_vehicle_available_moe,total_households,total_households_moe
0,5000100325,Addison town,0.612245,0.819537,3,4,490,58
1,5000108575,Bridport town,1.509434,1.344323,8,7,530,88
2,5000109025,Bristol town,6.125739,3.896236,114,72,1861,140
3,5000116000,Cornwall town,1.016260,1.638063,5,8,492,96
4,5000126300,Ferrisburgh town,2.949309,3.326007,32,36,1085,85


In [10]:
# percent of occupied housing units that are renter-occupied

# rename columns
df_renter_occupied = df_renter_occupied.rename(
    columns={
        "Estimate!!Total:!!Renter occupied": "renter_occupied",
        "Margin of Error!!Total:!!Renter occupied": "renter_occupied_moe",
        "Estimate!!Total:": "total_occupied",
        "Margin of Error!!Total:": "total_occupied_moe",
    }
)

# calculate percent renter-occupied
df_renter_occupied["pct_renter_occupied"] = (
    df_renter_occupied["renter_occupied"] / df_renter_occupied["total_occupied"] * 100
)

# calculate MOE for percent renter-occupied using ratio MOE formula
# MOE_percent = 100 * sqrt( (renter_occupied_moe/total_occupied)^2 + (renter_occupied*total_occupied_moe/total_occupied^2)^2 )
df_renter_occupied["pct_renter_occupied_moe"] = ratio_moe(
    df_renter_occupied["renter_occupied"],
    df_renter_occupied["renter_occupied_moe"],
    df_renter_occupied["total_occupied"],
    df_renter_occupied["total_occupied_moe"],
)

# keep only relevant columns
df_renter_occupied = df_renter_occupied[
    [
        "GEOID",
        "town_name",
        "pct_renter_occupied",
        "pct_renter_occupied_moe",
        "renter_occupied",
        "renter_occupied_moe",
        "total_occupied",
        "total_occupied_moe",
    ]
]
df_renter_occupied.head()

,GEOID,town_name,pct_renter_occupied,pct_renter_occupied_moe,renter_occupied,renter_occupied_moe,total_occupied,total_occupied_moe
0,5000100325,Addison town,10.816327,5.260227,53,25,490,58
1,5000108575,Bridport town,17.924528,9.354012,95,47,530,88
2,5000109025,Bristol town,28.694250,8.033232,534,144,1861,140
3,5000116000,Cornwall town,5.691057,2.866134,28,13,492,96
4,5000126300,Ferrisburgh town,14.654378,6.734520,159,72,1085,85
